# Integrated CLASP Pipeline

Tune parameters, build a CLASP graph, compare UMAP and IMSTE embeddings, save the embedded AnnData file, and plot both paired batch/label views.


In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

# Make the src-layout package importable when this notebook runs from the repo
# root or its notebooks/ directory, even if clasp is not installed editable.
_project_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "src" / "clasp").is_dir()),
    None,
)
if _project_root is None:
    raise RuntimeError("Could not find the CLASP project root from the notebook kernel working directory.")
_src_path = str(_project_root / "src")
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)

for name in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
):
    os.environ[name] = "1"
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("KMP_INIT_AT_FORK", "FALSE")

from clasp import (
    ClaspEstimator,
    ClaspGraphEmbedder,
    LabelNeighborGraph,
    EstimatorTuneParams,
    GraphTuneParams,
    LatentBOTuneParams,
    PreprocessTuneParams,
    TuneParams,
)
from clasp.notebook_utils import (
    dataset_config,
    list_dataset_config_files,
    optimized_params_path,
    preprocess_params,
)
from imste import IteratedMinimumSpanningTreeEmbedder


Restart the kernel before running this notebook if native numerical libraries have already been imported. The first cell sets conservative OpenMP/thread settings.

In [2]:
available_datasets = sorted(path.stem for path in list_dataset_config_files())
print("Available datasets:")
for dataset_name in available_datasets:
    print(f"- {dataset_name}")



Available datasets:
- cellrank_bone_marrow
- cellrank_lung
- cellrank_pancreas
- cellrank_reprogramming_morris
- cellrank_reprogramming_schiebinger
- pancreas
- pbmc3k
- scib_immune_human
- scib_immune_human_mouse
- scib_lung_atlas
- scib_pancreas
- zebrafish


In [3]:
selected_dataset = "scib_immune_human_mouse"
random_state = 0
run_tuning = True

# Optimization uses a small stratified subset. Increase this after the pipeline works end to end.
tuning_max_cells = 1000

# Final visualization can use more cells than optimization.
final_max_cells = 20_000

# Tune the CLASP graph and compare UMAP and IMSTE embeddings.
embedding_method = "umap"
embedding_epochs = 500
imste_n_msts = 256
imste_n_epochs = 2000

dataset = dataset_config(selected_dataset)
estimator = ClaspEstimator(
    batch_key=dataset["batch_key"],
    label_key=dataset["label_key"],
    n_components=100,
    random_state=random_state,
    embedder=ClaspGraphEmbedder(
        method=embedding_method,
        n_components=2,
        random_state=random_state,
        kwargs={"n_epochs": embedding_epochs},
    ),
)
raw_adata = estimator.to_data(dataset["input_path"])
raw_adata


AnnData object with n_obs × n_vars = 97861 × 8135
    obs: 'batch', 'chemistry', 'data_type', 'dpt_pseudotime', 'final_annotation', 'mt_frac', 'n_counts', 'n_genes', 'size_factors', 'species', 'study', 'tissue'
    layers: 'counts'

In [4]:
base_preprocess = preprocess_params(dataset, n_top_genes=1500)
fixed_preprocess = {"max_cells": tuning_max_cells}
base_estimator = {"n_components": 60}
base_graph = {
    **dataset.get("graph", {}),
    "metric": "euclidean",
}

preprocess_search_space = {
    "n_top_genes": {"type": "int", "bounds": [500, 2000]},
}
estimator_search_space = {
    "n_components": {"type": "int", "bounds": [20, 100]},
}
graph_search_space = {
    "n_neighbors": {"type": "int", "bounds": [5, 35]},
    "intra_fraction": {"type": "float", "bounds": [0.2, 0.9]},
    "n_inter_edges": {"type": "int", "bounds": [1, 6]},
    "assignment_quantile": {"type": "float", "bounds": [0.1, 1.0]},
    "hubness_k": {"type": "int", "bounds": [3, 20]},
    "rank_correction": {"type": "categorical", "values": [False, True]},
    "edge_weighting": {"type": "categorical", "values": ["binary", "distance"]},
    "inter_edge_mode": {"type": "categorical", "values": ["propagate_neighbors", "assignment"]},
    "mutual_neighbors": {"type": "categorical", "values": [False, True]},
}

tune_params = TuneParams(
    preprocess=PreprocessTuneParams(
        base=base_preprocess,
        fixed=fixed_preprocess,
        search_space=preprocess_search_space,
    ),
    estimator=EstimatorTuneParams(
        base=base_estimator,
        search_space=estimator_search_space,
    ),
    graph=GraphTuneParams(
        base=base_graph,
        search_space=graph_search_space,
    ),
    pca_bo=LatentBOTuneParams(
        n_initial=6,
        latent_dim=4,
        n_iterations=24,
        embedding_model="pca",
        acquisition="ei",
        batch_size=1,
    ),
    run_gplvm_refinement=False,
    random_state=random_state,
    embedding_method=embedding_method,
    embedding_epochs=embedding_epochs,
)


In [5]:
if run_tuning:
    tune_result = estimator.tune(raw_adata, tune_params)
    params_path = tune_result.save(optimized_params_path(selected_dataset))
else:
    tune_result = None
    params_path = optimized_params_path(selected_dataset)

import json
payload = json.loads(params_path.read_text())
metadata = payload.get("metadata", {})
optimized_params = {
    "params_path": str(params_path),
    "best_model": metadata.get("best_model"),
    "best_score": metadata.get("best_score"),
    "preprocess_params": payload.get("preprocess_params", {}),
    "estimator_params": payload.get("estimator_params", {}),
    "graph_params": payload.get("graph_params", {}),
}
optimized_params

/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


{'params_path': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/clasp/data/optimized_params/scib_immune_human_mouse_graph_params.json',
 'best_model': 'pca',
 'best_score': 0.73345,
 'preprocess_params': {'create_artificial_batch': False,
  'hvg_flavor': 'variance',
  'max_cells': 1000,
  'min_cell_genes': None,
  'min_gene_counts': 0,
  'n_top_genes': 500,
  'normalize': False},
 'estimator_params': {'n_components': 40},
 'graph_params': {'assignment_quantile': 0.1,
  'edge_weighting': 'distance',
  'hubness_correction': 'csls',
  'hubness_k': 3,
  'inter_edge_mode': 'assignment',
  'intra_fraction': 0.564145193135708,
  'metric': 'euclidean',
  'mutual_neighbors': True,
  'n_inter_edges': 4,
  'n_neighbors': 35,
  'neighbor_mode': 'distance',
  'rank_correction': True,
  'symmetrize': True}}

In [8]:
tuned_preprocess = payload.get("preprocess_params", dataset.get("preprocess", {}))
tuned_estimator = payload.get("estimator_params", {})
tuned_graph = payload.get("graph_params", dataset.get("graph", {}))

# Final run can use more cells than optimization while preserving tuned settings.
final_preprocess = {**tuned_preprocess, "max_cells": final_max_cells}
estimator = ClaspEstimator(
    batch_key=dataset["batch_key"],
    label_key=dataset["label_key"],
    random_state=random_state,
    **tuned_estimator,
)
adata = estimator.preprocess(raw_adata, **final_preprocess)
# CLASP's graph stores affinities for UMAP; IMSTE needs distance-valued edges.
clasp_graph = estimator.data_to_graph(adata, **tuned_graph)

imste_embedder = IteratedMinimumSpanningTreeEmbedder(
    n_msts=imste_n_msts,
    n_components=2,
    n_epochs=imste_n_epochs,
    random_state=random_state,
)
imste_graph_params = {**tuned_graph, "edge_weighting": "distance"}
if tuned_graph.get("edge_weighting", "distance") == "distance":
    imste_affinity_graph = clasp_graph
else:
    # Binary weights retain topology but discard the edge distances IMSTE needs.
    imste_affinity_graph = estimator.data_to_graph(adata, **imste_graph_params)
imste_estimator = ClaspEstimator(
    batch_key=dataset["batch_key"],
    label_key=dataset["label_key"],
    random_state=random_state,
    edge_weighting="distance",
    embedder=imste_embedder,
)
adata.obsm["X_clasp_imste"] = imste_estimator.graph_to_embeddings(imste_affinity_graph)

umap_embedder = ClaspGraphEmbedder(
    method="umap",
    n_components=2,
    random_state=random_state,
    kwargs={"n_epochs": embedding_epochs},
)
umap_embedder.fit_graph(clasp_graph)
adata.obsm["X_clasp_umap"] = umap_embedder.embedding_

estimator.save(adata, dataset["output_path"])
adata


AnnData object with n_obs × n_vars = 20000 × 500
    obs: 'batch', 'chemistry', 'data_type', 'dpt_pseudotime', 'final_annotation', 'mt_frac', 'n_counts', 'n_genes', 'size_factors', 'species', 'study', 'tissue'
    var: 'clasp_hvg_score', 'clasp_selected'
    uns: 'clasp'
    obsm: 'X_pca', 'X_clasp_imste', 'X_clasp_umap'
    layers: 'counts'

In [ ]:
RANDOMIZE_LABEL_COLORS = True
LABEL_COLOR_SEED = 42
label_series = adata.obs[dataset["label_key"]]
label_strings = label_series.astype(str)
color_label_order = sorted(label_strings.unique().tolist())
label_palette = list(sns.color_palette("tab20", n_colors=len(color_label_order)))
if RANDOMIZE_LABEL_COLORS:
    color_permutation = np.random.default_rng(LABEL_COLOR_SEED).permutation(len(label_palette))
    label_palette = [label_palette[index] for index in color_permutation]
LABEL_COLORS = dict(zip(color_label_order, label_palette))
if label_series.isna().any():
    missing_strings = label_series[label_series.isna()].astype(str).unique()
    if len(missing_strings):
        LABEL_COLORS["Missing"] = LABEL_COLORS[str(missing_strings[0])]

estimator.plot(
    adata,
    embedding_key="X_clasp_umap",
    s=8,
    alpha=0.85,
    random_state=random_state,
    legend_markerscale=3.5,
    label_palette=LABEL_COLORS,
    filename=f"figures/{selected_dataset}_integrated_pipeline_umap.pdf",
);

estimator.plot(
    adata,
    embedding_key="X_clasp_imste",
    s=8,
    alpha=0.85,
    random_state=random_state,
    legend_markerscale=3.5,
    label_palette=LABEL_COLORS,
    filename=f"figures/{selected_dataset}_integrated_pipeline_imste.pdf",
);


In [ ]:
K_LABEL_NEIGHBORS = 30
SHOW_SELF_LINKS = False
SHOW_NODE_COUNTS = False
NODE_ALPHA = 0.7
MIN_NODE_SIZE = 1000
MAX_NODE_SIZE = 2500
MIN_EDGE_FRACTION = 0.15

graph_labels = adata.obs[dataset["label_key"]]
label_positions_path = _project_root / "figures" / f"{selected_dataset}_label_positions.json"
node_position_selector = widgets.Dropdown(
    options=[("UMAP", "umap"), ("IMSTE", "imste")],
    value="umap",
    description="Node positions:",
)
display(node_position_selector)

def plot_label_transition_graphs(node_position_source):
    embeddings = {
        "umap": adata.obsm["X_clasp_umap"],
        "imste": adata.obsm["X_clasp_imste"],
    }
    position_embedding = embeddings[node_position_source]
    umap_label_graph = LabelNeighborGraph(
        k=K_LABEL_NEIGHBORS,
        min_edge_fraction=MIN_EDGE_FRACTION,
        min_node_size=MIN_NODE_SIZE,
        max_node_size=MAX_NODE_SIZE,
    ).fit(embeddings["umap"], graph_labels)
    imste_label_graph = LabelNeighborGraph(
        k=K_LABEL_NEIGHBORS,
        min_edge_fraction=MIN_EDGE_FRACTION,
        min_node_size=MIN_NODE_SIZE,
        max_node_size=MAX_NODE_SIZE,
    ).fit(embeddings["imste"], graph_labels)

    label_colors = LABEL_COLORS

    # Position both graph panels using label centroids from the selected embedding.
    umap_label_graph.set_positions_from_embedding(position_embedding)
    umap_label_graph.export_positions(label_positions_path)
    umap_label_graph.import_positions(label_positions_path)
    imste_label_graph.import_positions(label_positions_path)

    fig, axes = plt.subplots(1, 2, figsize=(18, 9))
    umap_label_graph.plot(
        ax=axes[0],
        show_self_links=SHOW_SELF_LINKS,
        show_node_counts=SHOW_NODE_COUNTS,
        node_alpha=NODE_ALPHA,
        label_colors=label_colors,
        title=f"UMAP label transitions (k={K_LABEL_NEIGHBORS})",
    )
    imste_label_graph.plot(
        ax=axes[1],
        show_self_links=SHOW_SELF_LINKS,
        show_node_counts=SHOW_NODE_COUNTS,
        node_alpha=NODE_ALPHA,
        label_colors=label_colors,
        title=f"IMSTE label transitions (k={K_LABEL_NEIGHBORS})",
    )
    fig.suptitle(f"Shared node positions from {node_position_source.upper()}")
    fig.text(
        0.5,
        0.025,
        "Node area is proportional to label count; arrow thickness is the mean fraction of k-NN cells from source to destination label.",
        ha="center",
    )
    fig.subplots_adjust(bottom=0.09, wspace=0.18, top=0.9)
    plt.show()

node_position_output = widgets.interactive_output(
    plot_label_transition_graphs,
    {"node_position_source": node_position_selector},
)
display(node_position_output)
